In [ ]:
import json
import pandas as pd
import os
from pathlib import Path

# Run this notebook from within the 09_llm_categorization/ directory.
# All intermediate JSONL files are saved here; input/output CSVs use relative paths.
notebook_dir = Path.cwd()
project_root = notebook_dir.parent

data = pd.read_csv(project_root / "data" / "raw" / "literature_dataset.csv")
dataShort = data

In [ ]:
def parse_keywords(row):
    # Get Title and Abstract, treating NaN as an empty string
    parsed_input = (row['Title'] if pd.notna(row['Title']) else '') + " " + (row['Abstract'] if pd.notna(row['Abstract']) else '')
    
    # Check for keywords in the specified order and add the first non-empty one found
    if pd.notna(row['Author Keywords']) and row['Author Keywords'].strip():
        parsed_input += " " + row['Author Keywords']
    elif pd.notna(row['WOS Keywords']) and row['WOS Keywords'].strip():
        parsed_input += " " + row['WOS Keywords']
    elif pd.notna(row['Scopus Keywords']) and row['Scopus Keywords'].strip():
        parsed_input += " " + row['Scopus Keywords']

    return parsed_input

# Apply the function row by row
# dataShort['parsedInput'] = dataShort.apply(parse_keywords, axis=1)

# def parse_keywords(row):
#     # Get Title and Abstract, treating NaN as an empty string
#     parsed_input = (row['Title'] if pd.notna(row['Title']) else '') + " " + (row['Abstract'] if pd.notna(row['Abstract']) else '')
    
#     # Check for keywords in the specified order and add the first non-empty one found
#     if pd.notna(row['Author Keywords']) and row['Author Keywords'].strip():
#         parsed_input += " " + row['Author Keywords']
#     elif pd.notna(row['Scopus Keywords']) and row['Scopus Keywords'].strip():
#         parsed_input += " " + row['Scopus Keywords']
#     elif pd.notna(row['WOS Keywords']) and row['WOS Keywords'].strip():
#         parsed_input += " " + row['WOS Keywords']
    
#     return parsed_input

# Apply the function row by row
dataShort['parsedInput'] = dataShort.apply(parse_keywords, axis=1)

In [ ]:
# List to hold all API call objects
batch_requests = []
for idx, parsedText in enumerate(dataShort['parsedInput'], start=1):
    batch_requests.append({
        "custom_id": f"{dataShort['ID'][idx-1]}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            # "model": "gpt-5-mini", # choose one of the two
            "model": "o4-mini", # choose one of the two
            "reasoning_effort": "medium", # that's default
            "n": 5, # maybe go up to 10 for the final run?
            "temperature": 1, # would do smaller for non-reasoning models, but reasoning models are 1 by default
            "messages": [
                    {
                        "role": "developer",
                        "content": "You are an expert at reviewing papers in the social science related to air pollution."
                    },
                    {
                        "role": "user",
                        "content": f"""
                This is the title, abstract, and keywords for a given research paper: 
                ```{parsedText}```

                Please classify this paper as to be included or excluded from our study, based on the following criteria:

                1. It is a research study that uses survey data related to air pollution or air quality as a primary research method (including primary, secondary and experimental survey types). The study can be mixed methods (e.g. include some qualitative work), but needs to at least also include survey data.

                2. The study needs to focus on air pollution either as the primary outcome (dependent variable) or as explanatory variable. It can also include air quality perceptions, but this needs to focus on pollution aspects (e.g. perceptions of air quality unrelated to air pollution such as testing air conditioning or ventilation systems or assessing thermal comfort of buildings do NOT meet the criteria).

                3. The study can focus on either outdoor or indoor air pollution, or both.

                If the paper does not meet ALL these criteria, it should be classified as excluded.

                Steps to follow:
                1. Write a very short explanation (one sentence) of why you would include or exclude that paper from our study.
                2. On the next line write "output_json".
                3. On the following line provide only the JSON object with key "inclusion" and value either "included" or "excluded", for example:
                {{
                "inclusion": ["included"]
                }}
                """
                    }
                ],
            # "max_tokens": 300 # if using gpt-4.1-mini or similar
            'max_completion_tokens': 800, # if using o4-mini or similar reasoning model
        }
    })

with open(notebook_dir / "batch_requests_inclusion_o4mini_final.jsonl", "w", encoding="utf-8") as fout:
    for req in batch_requests:
        fout.write(json.dumps(req) + "\n")

In [ ]:
from openai import OpenAI

client = OpenAI(api_key="INSERT-YOUR-KEY-HERE")

In [ ]:
batch_input_file = client.files.create(
    file=open("batch_requests_inclusion_o4mini_final.jsonl", "rb"),
    purpose="batch"
)

print(batch_input_file)

In [ ]:
batch_input_file_id = batch_input_file.id
batch_created = client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={
        "description": "inclusion papers o4mini n5 fullCall",
    }
)

print(batch_created)
print(batch_created.id)
batch_id = batch_created.id

In [ ]:
# 2) Retrieve the batch by its ID
# batch = client.batches.retrieve('batch_68b6c1fa83dc819088b5173277490ff0')
batch = client.batches.retrieve(batch_id)

# 3) Print the full Batch object
print(batch)

# extract the batch ID and output file id from the response
batch_id = batch.id
output_file_id = batch.output_file_id

error_file_id = batch.error_file_id

In [ ]:
# check for error
file_error = client.files.content(error_file_id) # latest batch_68b6c1fa83dc819088b5173277490ff0
file_error.text

In [ ]:
file_response = client.files.content(output_file_id) # latest batch_68b6c1fa83dc819088b5173277490ff0
print(file_response.text)

file_response.write_to_file("batch_requests_inclusion_o4mini_final_response.jsonl")

In [ ]:
# import json
# import pandas as pd

# 1. Read your JSONL
with open("batch_requests_inclusion_o4mini_final_response.jsonl", "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f]

# 2. Normalize, exploding the choices list into rows
df = pd.json_normalize(
    records,
    record_path=["response", "body", "choices"],
    meta=[
        "custom_id",
        ["response", "body", "model"],
        ["response", "body", "created"],
        ["id"]
    ],
    errors="ignore"
)

# 4. Clean up column names if you like
df = df.rename(
    columns={
        "index": "choice_index",
        "message.content": "message_content"
    }
)

print(df)

In [ ]:
import re

def extract_identification(text):
    # Use regular expressions to find the JSON portion of the text
    json_match = re.search(r'\{.*?\}', text, re.DOTALL)
    
    if json_match:
        # Extract the JSON string
        json_str = json_match.group(0)
        
        # Load the JSON string into a Python dictionary
        json_data = json.loads(json_str)
        
        # Extract the value of the 'countries' key
        classification = json_data.get("inclusion", None)
        return str(classification)
    else:
        return None


In [ ]:
df['inclusionChatGPT'] = df['message_content'].apply(extract_identification)
df['inclusionChatGPTClean'] = df['inclusionChatGPT'].str.replace(r"[\[\]'\"]", "", regex=True)

In [ ]:
# select important columns and save to csv
df_save = df[['custom_id', "choice_index", 'message_content', 'inclusionChatGPTClean', 'response.body.model']]

df_save.to_csv(project_root / "data" / "llm_classifications" / "inclusion_o4mini.csv", index=False)